# 05 Machine Learning Models

Train tabular regression models for temperature prediction, compare results, and save the best model.


In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "GlobalWeatherRepository.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

from src.features import prepare_model_frame
from src.preprocessing import load_processed_data
from src.train import compare_regression_models, save_model
from src.visualize import save_figure


In [ ]:
anomaly_free_path = PROCESSED_DIR / "weather_without_anomalies.csv"
clean_path = PROCESSED_DIR / "clean_weather_data.csv"
source_path = anomaly_free_path if anomaly_free_path.exists() else clean_path

df = load_processed_data(source_path)
X, y = prepare_model_frame(df, target_column="temperature_celsius")

print(f"Loaded source data from: {source_path}")
print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
display(X.head())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

models, results_df = compare_regression_models(X_train, X_test, y_train, y_test)
display(results_df)


In [ ]:
best_model_name = results_df.index[0]
best_model = models[best_model_name]
best_model_path = MODELS_DIR / f"{best_model_name}_temperature_model.joblib"
save_model(best_model, best_model_path)

metrics_path = PROJECT_ROOT / "reports" / "temperature_model_metrics.csv"
results_df.to_csv(metrics_path)

print(f"Best model: {best_model_name}")
print(f"Saved best model to: {best_model_path}")
print(f"Saved comparison metrics to: {metrics_path}")


In [ ]:
if hasattr(best_model, "feature_importances_"):
    feature_importance = (
        pd.Series(best_model.feature_importances_, index=X_train.columns)
        .sort_values(ascending=False)
        .head(15)
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    feature_importance.plot(kind="bar", ax=ax, color="teal")
    ax.set_title(f"Top Feature Importances ({best_model_name})")
    ax.set_ylabel("Importance")
    plt.tight_layout()
    display(fig)
    save_figure(fig, FIGURES_DIR / "ml_feature_importance.png")
    plt.close(fig)
else:
    print(f"{best_model_name} does not expose feature_importances_.")
